In [1]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

In [2]:
# imports
import glob
from pprint import pprint
from tqdm import tqdm
from collections import Counter
from src.const.llm import ModelAPIConfig
from src.kgqa_tool.llm_request import check_early_stop

/scratch/hpc-prf-merlin/nikit/repos/ag-rag-kgqa/venv_ag-rag-kgqa_py39/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm_config = ModelAPIConfig("gpt-oss-120b", 'http://lola.cs.uni-paderborn.de:9292/v1', '') # LLM to use

In [4]:
# experiment analysis dictionary
analysis_path_dict = {
    'qald9plus_test': ['data_dir/processed_kgqa_ds/qald9plus/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b'],
    'qald10_test': ['data_dir/processed_kgqa_ds/qald10/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b'],
    'lcquad2_test': ['data_dir/processed_kgqa_ds/lcquad2/test/test.prediction/tentrisq10_aug_gold/analysis/en__lola__PBSG_MHOP__t20-h-1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b']
}

In [5]:
multilingual_data = {
    'qald9plus_test': {
        'template': 'data_dir/processed_kgqa_ds/qald9plus/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/${lang}__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b',
        'langs': ['ba', 'be', 'de', 'es', 'fr', 'hy', 'lt', 'ru', 'uk']
    },
    'qald10_test': {
        'template': 'data_dir/processed_kgqa_ds/qald10/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/${lang}__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b',
        'langs': ['de', 'ru', 'zh']
    }
    # (lcquad2_test has no multilingual logs, so we omit it)
}

def expand_multilingual_paths(path_dict, multi_data):
    """
    Extend ``path_dict`` with language‑specific analysis directories
    derived from the ``multi_data`` templates.
    """
    for dataset, meta in multi_data.items():
        tmpl = meta['template']
        for lang in meta['langs']:
            # Substitute the language placeholder
            path = tmpl.replace('${lang}', lang)
            # Ensure the dataset key exists
            path_dict.setdefault(dataset, [])
            # Append if not already present
            if path not in path_dict[dataset]:
                path_dict[dataset].append(path)

# Apply the expansion once, before any analysis runs
expand_multilingual_paths(analysis_path_dict, multilingual_data)
pprint(analysis_path_dict, width=120)

{'lcquad2_test': ['data_dir/processed_kgqa_ds/lcquad2/test/test.prediction/tentrisq10_aug_gold/analysis/en__lola__PBSG_MHOP__t20-h-1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b'],
 'qald10_test': ['data_dir/processed_kgqa_ds/qald10/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b',
                 'data_dir/processed_kgqa_ds/qald10/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/de__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b',
                 'data_dir/processed_kgqa_ds/qald10/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/ru__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b',
                 'data_dir/processed_kgqa_ds/qald10/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/zh__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b'],
 'qald9plus_test': ['data_dir/processed_kgqa_ds/qald9plus/test/ablation.test.prediction/te

In [6]:
precomputed_stats = {
    # from a previous run: lcquad2_test: 1257/2515 early‑stops : 49.98%
    'lcquad2_test' : {"total": 2515,  "early": 1257, "percent": 49.98},
}

In [7]:
def is_early_stop(content):
    llm_resp, think_content = check_early_stop(content, llm_config)
    #print(f'{think_content}\n{llm_resp}\n\n')
    return llm_resp.lower() == 'y'

In [8]:
def analyse_dataset(dataset_name: str, analysis_dirs: list) -> dict:
    """
    Walks through every ``*_analysis.txt`` file in the supplied directories,
    runs ``check_early_stop`` on the file content and returns a dict with:

        {
            "total":   <number of analysis files examined>,
            "early":   <number flagged as early termination>,
            "percent": <early / total * 100 (float)>,
        }
    """
    total, early = 0, 0

    for rel_dir in analysis_dirs:
        abs_dir = get_fp(rel_dir)               # absolute path to the dir
        # Grab every file that ends with *_analysis.txt (recursively)
        pattern = os.path.join(abs_dir, "**", "*_analysis.txt")
        for file_path in tqdm(glob.glob(pattern, recursive=True),
                              desc=f"Scanning {dataset_name} :: {rel_dir}",
                              unit="file"):
            total += 1
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            try:
                if is_early_stop(content):
                    early += 1
            except Exception as e:
                print(f"is_early_stop failed for {file_path}: {e}")

    percent = (early / total * 100) if total else 0.0
    return {"total": total, "early": early, "percent": percent}

In [9]:

# Main aggregation over all datasets
dataset_stats = {}
overall_total, overall_early = 0, 0
macro_percent_sum = 0.0   # for macro‑average (simple mean of per‑dataset percents)

for ds_name, dirs in analysis_path_dict.items():
    if ds_name in precomputed_stats:
        # Use the stats we already have – no need to run analyse_dataset()
        stats = precomputed_stats[ds_name]
    else:
        # Fall back to the original computation
        stats = analyse_dataset(ds_name, dirs)
    dataset_stats[ds_name] = stats

    overall_total += stats["total"]
    overall_early += stats["early"]
    macro_percent_sum += stats["percent"]

# Micro‑average (global early‑stop rate)
micro_percent = (overall_early / overall_total * 100) if overall_total else 0.0

# Macro‑average (average of per‑dataset percentages)
macro_percent = (macro_percent_sum / len(dataset_stats)) if dataset_stats else 0.0

# Pretty‑print the results
print("\n=== Early‑Termination Analysis ===\n")
for ds, stats in dataset_stats.items():
    print(f"\n{ds:>12}: {stats['early']}/{stats['total']} "
          f"early‑stops : {stats['percent']:.2f}%")

print("\nOverall (micro) early‑stop rate: "
      f"{overall_early}/{overall_total} : {micro_percent:.2f}%")
print("Overall (macro) early‑stop rate: "
      f"{macro_percent:.2f}%\n")

Scanning qald9plus_test :: data_dir/processed_kgqa_ds/qald9plus/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b: 100%|██████████| 64/64 [01:22<00:00,  1.29s/file]
Scanning qald9plus_test :: data_dir/processed_kgqa_ds/qald9plus/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/ba__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b: 100%|██████████| 60/60 [01:14<00:00,  1.24s/file]
Scanning qald9plus_test :: data_dir/processed_kgqa_ds/qald9plus/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/be__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b: 100%|██████████| 72/72 [01:42<00:00,  1.43s/file]
Scanning qald9plus_test :: data_dir/processed_kgqa_ds/qald9plus/test/multilingual.test.prediction/tentrisq10_aug_gold/analysis/de__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b: 100%|██████████| 70/70 [01:51<00:00,  1.60s/file]



=== Early‑Termination Analysis ===


qald9plus_test: 303/562 early‑stops : 53.91%

 qald10_test: 481/809 early‑stops : 59.46%

lcquad2_test: 1257/2515 early‑stops : 49.98%

Overall (micro) early‑stop rate: 2041/3886 : 52.52%
Overall (macro) early‑stop rate: 54.45%

